In [34]:
print("hello")

hello


In [35]:
from os import path
main_path='t2-ragbench/data/ConvFinQA'

json_path=path.join(main_path,'turn_0.jsonl')


In [36]:
import json

def Load_josn(p):
    with open(p, 'r', encoding='utf-8') as f:
        d = json.load(f)
    return d


pdf_urls=Load_josn('data_set\\pdf_urls.json')
question_doc = Load_josn('data_set\\qrels.json')
queries=Load_josn('data_set\\queries.json')
answers=Load_josn('data_set\\answers.json')



In [96]:
import pandas as pd
df = pd.DataFrame.from_dict(question_doc, orient='index')
df.index.name = 'query_id'
df = df.reset_index()  

print(df)

                                  query_id        doc_id  section_id
0     852703f0-8373-43a2-a18a-eb5908ad0779  2410.14077v2           1
1     9199173b-3ed1-4118-88cd-1713fc5fa8a7  2404.00822v2          17
2     1d585069-a446-47fa-a74d-0387316ea330  2410.07168v2          30
3     dc064d11-cd18-4866-8a99-f16b0abec9c6  2401.07294v4          12
4     283afa84-f0c8-40a7-a6f1-fb2a6b97c761  2411.14884v3           1
...                                    ...           ...         ...
3040  08a34950-7004-433d-ac0e-8c48363ce406  2410.23587v3           2
3041  029ae88b-dc53-4f88-918b-c8478d7ef0d3  2410.12710v2           2
3042  d369250a-8506-4c6c-948b-e9383e88e0e2  2410.10516v3          15
3043  6d7c948c-4d17-4b27-bb83-eb2b88728035  2408.02322v2           0
3044  90144eed-61dd-4a4a-b85d-0e01d597004a  2403.12117v2          27

[3045 rows x 3 columns]


In [100]:
doc_list=df.groupby('doc_id').count().sort_values(by='query_id',ascending=False).head(10).index.tolist()
df=df[df['doc_id'].isin(doc_list)]

In [83]:
import requests, os
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs("pdfs", exist_ok=True)
items ={}
for i ,j in pdf_urls.items():
    if(i in doc_list):
        items[i]=j
def download(name_url):
    name, url = name_url
    try:
        res = requests.get(url, timeout=30)
        res.raise_for_status()
        fname = os.path.join("pdfs", f"{name}.pdf")
        with open(fname, "wb") as f:
            f.write(res.content)
        return name, True, None
    except Exception as e:
        return name, False, str(e)

pdfs_ = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(download, item) for item in items.items()]
    for future in as_completed(futures):
        name, ok, err = future.result()
        if ok:
            pdfs_.append(name)
        else:
            print(f"failed: {name} -> {err}")

print(f"{len(pdfs_)}/{len(items)} downloaded")

10/10 downloaded


In [104]:
q_df = pd.DataFrame.from_dict(queries, orient='index')  # columns: query, type, source
q_df.index.name = 'query_id'
q_df = q_df.reset_index().rename(columns={'type': 'query_type', 'source': 'query_source', 'query': 'query'})

df = df.merge(q_df, on='query_id', how='left')

In [109]:
answer_df=pd.DataFrame.from_dict(answers,orient="index")
answer_df.index.name='query_id'
df=df.merge(answer_df,on='query_id',how='left')

In [115]:
df=df.rename(columns={'Answer':'answer'})
df

,query_id,doc_id,section_id,query,query_type,query_source,answer
0,dc064d11-cd18-4866-8a99-f16b0abec9c6,2401.07294v4,12,How does the MLMM approach affect the analysis...,abstractive,text-image,The MLMM approach affects the analysis of RMSE...
1,bac61451-d99a-43b3-9754-b8a593e5d1d7,2401.11899v3,10,Does bounded invariance affect how probability...,extractive,text,"No, bounded invariance states that changes in ..."
2,70ef4593-1b52-42c9-8285-27793e5bd538,2412.20317v3,23,How can graph drawing methods be combined to h...,abstractive,text-image,Graph drawing methods can be combined with tec...
3,769e2fab-e157-4e4a-a0a1-b7b2e621a3d3,2501.00225v2,0,How are complexified tetrahedrons used in knot...,abstractive,text,"Complexified tetrahedrons, which have complex ..."
4,0b4acbde-fe78-419f-930a-04d49c0630b6,2401.11899v3,15,How does the recursive algorithm ensure that a...,abstractive,text-table,"The recursive algorithm uses supply vectors, p..."
...,...,...,...,...,...,...,...
95,15f75010-c87a-4c3e-b471-d969681c3cab,2412.20317v3,7,What is the Newton direction in the context of...,extractive,text,The Newton direction is given by \(d=-\nabla^{...
96,ca35d598-a95d-4618-a2f0-21a8bd24545b,2401.07294v4,0,What theoretical rationale supports using mult...,abstractive,text,The theoretical rationale for using MLMM lies ...
97,37e267c7-f836-468f-8f82-69ace5ac7e55,2401.06987v2,13,What is the formula for absolute sensitivity i...,extractive,text,The formula for absolute sensitivity in the ca...
98,f8ca8e45-3ee3-448d-aa95-4bc20f130b4e,2401.07317v2,5,Is the new definition of $\mathbb{B}$-space ex...,extractive,text,Yes.


In [116]:
df.head(2)

,query_id,doc_id,section_id,query,query_type,query_source,answer
0,dc064d11-cd18-4866-8a99-f16b0abec9c6,2401.07294v4,12,How does the MLMM approach affect the analysis...,abstractive,text-image,The MLMM approach affects the analysis of RMSE...
1,bac61451-d99a-43b3-9754-b8a593e5d1d7,2401.11899v3,10,Does bounded invariance affect how probability...,extractive,text,"No, bounded invariance states that changes in ..."


In [ ]:
from rag_eval.parsers import DoclingParser

DoclingParser